In [1]:
import gc
import h5py
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

from utils.dataloader import load_shards

In [2]:
# Function to calculate mean and std of training and testing sets
def calculate_stats(tf_dataset, num_samples=5000):
    images_iterator = tf_dataset.unbatch().take(num_samples).as_numpy_iterator()
    all_images = np.stack([img for img, _ in images_iterator])

    means = np.mean(all_images, axis=(0, 1, 2))
    stds = np.std(all_images, axis=(0, 1, 2))

    return means.tolist(), stds.tolist()

In [3]:
# Load the data
file_pattern = '../data/*.tfrecord.gz'
all_files = tf.io.gfile.glob(file_pattern)
print(len(all_files), 'shards loaded')

# Split training and testing data
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

1500 shards loaded


In [4]:
loader_batch_size = 32

# 1. Calculate stats
raw_for_stats = load_shards(train_files, batch_size=loader_batch_size)
train_means, train_stds = calculate_stats(raw_for_stats)

# FORCE PYTHON TO DUMP THE STATS DATA FROM RAM
del raw_for_stats
gc.collect()

# 2. Load streaming datasets
train_raw = load_shards(
    train_files, batch_size=loader_batch_size, flatten=True
)
test_raw = load_shards(
    test_files, batch_size=loader_batch_size, is_training=False, flatten=True
)

h5_filename = 'svm_ready_data.h5'

def export_dataset_to_h5(tf_dataset, h5_file, X_name, y_name):
    iterator = tf_dataset.as_numpy_iterator()
    first_X, first_y = next(iterator)
    feature_dim = first_X.shape[1]
    batch_size = first_X.shape[0]

    # MEMORY FIX: Added 'chunks' so HDF5 flushes directly to disk and clears RAM immediately
    X_ds = h5_file.create_dataset(
        X_name, shape=(0, feature_dim), maxshape=(None, feature_dim),
        dtype='float16', chunks=(batch_size, feature_dim)
    )
    y_ds = h5_file.create_dataset(
        y_name, shape=(0,), maxshape=(None,),
        dtype='int32', chunks=(batch_size,)
    )

    # Write first batch
    X_ds.resize(batch_size, axis=0)
    y_ds.resize(batch_size, axis=0)
    X_ds[0 : batch_size] = first_X.astype(np.float16)
    y_ds[0 : batch_size] = first_y

    current_size = batch_size

    for i, (X_batch, y_batch) in enumerate(iterator):
        b_size = X_batch.shape[0]

        X_ds.resize(current_size + b_size, axis=0)
        y_ds.resize(current_size + b_size, axis=0)

        X_ds[current_size : current_size + b_size] = X_batch.astype(np.float16)
        y_ds[current_size : current_size + b_size] = y_batch

        current_size += b_size

        # Periodically force python to clean up RAM every 100 batches
        if i % 100 == 0:
            gc.collect()

    print(f"Exported {current_size} total samples to '{X_name}'")

# Run the export
with h5py.File(h5_filename, 'w') as hf:
    print("Exporting Training Data...")
    export_dataset_to_h5(train_raw, hf, 'X_train', 'y_train')

    print("Exporting Testing Data...")
    export_dataset_to_h5(test_raw, hf, 'X_test', 'y_test')

print(f"All done! You can now download {h5_filename}")

Exporting Training Data...
Exported 47529 total samples to 'X_train'
Exporting Testing Data...
Exported 11673 total samples to 'X_test'
All done! You can now download svm_ready_data.h5
